# Classes and encapsulation

Classes become truly useful once you stop seeing them as “functions with extra syntax” and start seeing them as a way to protect and organise state. Encapsulation means that an object controls how its internal data is created, read, and changed.

In Python, encapsulation is more about clear conventions and controlled interfaces than about hard privacy barriers. Properties, methods, and careful attribute design help you express what outside code is allowed to depend on.

A strong question to ask in this module is: **what state should this object own, and what operations should be the only safe way to change it?**

## Visual model

```text
object state + methods that protect that state
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. The attribute-lookup ladder

**The single most useful diagram in Part 2.** When you write `obj.x`, Python:

```text
1. type(obj).__mro__  -- looking for a DATA DESCRIPTOR named x
                         (something with __get__ AND __set__ -- e.g. @property)
                         found? call its __get__ and STOP.
2. obj.__dict__['x']  -- the instance's own dictionary
                         found? return it and STOP.
3. type(obj).__mro__  -- the class and its bases, in MRO order
                         found? return it (binding it if it is a function)
4. type(obj).__getattr__('x')   -- last-resort hook, if defined
5. AttributeError
```


Two consequences that explain a great deal:

**Instance attributes shadow class attributes** (step 2 beats step 3) — but
**properties beat instance attributes** (step 1 beats step 2). That ordering is
what makes `@property` able to intercept an attribute that used to be plain
data.

**A method is found on the class, not the instance.** Every instance of a class
shares one function object; the binding happens at lookup time.

In [ ]:
class Dog:
    def speak(self): return "woof"

d = Dog()
Dog.speak            # <function Dog.speak>       -- a plain function
d.speak              # <bound method Dog.speak>   -- function + instance
d.speak()            # == Dog.speak(d)

That is all `self` is: the first parameter, filled in by the binding. Python
makes it explicit rather than implicit, which is why you can do this:

In [ ]:
Dog.speak(d)                      # call it unbound
handler = d.speak                 # store a bound method as a callback
list(map(str.upper, ["a", "b"]))  # use an unbound method as a function

---

## Concept 3. Class attributes versus instance attributes

In [ ]:
class Counter:
    count = 0                     # CLASS attribute -- one, shared

    def __init__(self):
        self.items = []           # INSTANCE attribute -- one per object

The trap, and it is Module 02 wearing a class costume:

In [ ]:
class Basket:
    contents = []                 # SHARED between every instance

a, b = Basket(), Basket()
a.contents.append("apple")        # MUTATES the shared list
print(b.contents)                 # ['apple']   <-- !

Whereas:

In [ ]:
a.contents = ["apple"]            # REBINDS: creates an INSTANCE attribute
print(b.contents)                 # []          -- b still sees the class one

Mutation hits the shared object; assignment creates a per-instance shadow. Same
two operations from Module 02, same opposite outcomes.

**Rule: mutable state goes in `__init__`.** Class attributes are for constants,
defaults that are immutable, and things genuinely shared by all instances (a
registry, a counter of instances created).

---

## Concept 5. `@property`: why Python has no getters

In Java you write getters from the start because changing a public field to a
method later breaks every caller. **In Python it does not**, because
`@property` intercepts attribute access at the same syntax.

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # start plain. No getter, no setter.

    @property
    def area(self) -> float:      # a computed, read-only attribute
        return 3.14159 * self.radius ** 2

c = Circle(2)
c.area                            # 12.56...   -- no parentheses
c.area = 5                        # AttributeError: property has no setter

Adding validation later, without changing any call site:

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # this now goes through the setter

    @property
    def radius(self) -> float:
        return self._radius

    @radius.setter
    def radius(self, value: float) -> None:
        if value <= 0:
            raise ValueError(f"radius must be positive, got {value}")
        self._radius = value

Every existing `c.radius` and `c.radius = 5` keeps working, now validated. This
is why **you should not write a getter and setter until you need one.** Start
with a plain attribute; promote it to a property when there is a reason.

Two things to watch:

**Infinite recursion.** Inside the property, use `self._radius`, never
`self.radius` — the latter calls the property again.

**Cheapness.** A property looks like an attribute, so callers assume it is
cheap. A property that issues a database query will be called in a loop by
someone who had no way to know. If it is expensive, make it a method named
`compute_x()`, or cache it:

In [ ]:
from functools import cached_property

class Dataset:
    @cached_property
    def stats(self) -> dict[str, float]:      # computed once, then stored
        return expensive_analysis(self.rows)  # in the instance __dict__

`cached_property` works by writing the result into `self.__dict__`, so step 2 of
the lookup ladder finds it on every subsequent access and the descriptor never
runs again. (Which means it needs a `__dict__` — it does not work with
`__slots__`.)

---

## Concept 7. `__slots__`

By default every instance carries a `__dict__`. `__slots__` replaces it with a
fixed array of named slots.

In [ ]:
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

p = Point(1, 2)
p.z = 3            # AttributeError: 'Point' object has no attribute 'z'

| | Without slots | With slots |
|---|---|---|
| Memory per instance | ~56 bytes + dict (~104+) | ~48 bytes |
| Attribute access | dict lookup | array index (slightly faster) |
| Add new attributes | yes | no |
| `__dict__` | yes | no |
| Multiple inheritance | free | restricted |
| `cached_property`, `weakref` | work | need explicit slots entries |

**Use it when you have many instances of a small, fixed-shape object**: points,
tokens, tree nodes, cache entries, parsed records. Typical saving is 40 to 50
percent of memory, which at 10⁶ instances is the difference between fitting in
RAM and not.

Do not reach for it by default. It removes flexibility that libraries
(especially mocking and serialization ones) sometimes rely on, and the memory
saving is meaningless at 10³ instances. `@dataclass(slots=True)` (Module 11)
gives you it without the boilerplate.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: A class body is executable code
- Section 2: The attribute-lookup ladder
- Section 3: Class attributes versus instance attributes
- Section 4: There is no `private`
- Section 5: `@property`: why Python has no getters
- Section 6: `@classmethod` and `@staticmethod`
- Section 7: `__slots__`
- Section 8: Encapsulation that actually works

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import sys
import timeit
import tracemalloc

---

## `PointDict`

_PointDict_

In [ ]:
class PointDict:
    def __init__(self, x: float, y: float, z: float) -> None:
        self.x, self.y, self.z = x, y, z

---

## `PointSlots`

_PointSlots_

In [ ]:
class PointSlots:
    __slots__ = ("x", "y", "z")

    def __init__(self, x: float, y: float, z: float) -> None:
        self.x, self.y, self.z = x, y, z

---

## `measure_single_instance`

Report sys.getsizeof for one instance of each, AND for its __dict__.

In [ ]:
def measure_single_instance() -> None:
    """Report sys.getsizeof for one instance of each, AND for its __dict__.

    TRAP: sys.getsizeof(obj) does NOT include the __dict__ it points at. Add
    them. Getting this wrong makes __slots__ look like it saves 8 bytes.

    Then answer: what does sys.getsizeof miss even after you add the dict?
    (Hint: what do the x, y and z attributes point AT?)
    """
    raise NotImplementedError

---

## `measure_at_scale`

Use tracemalloc to measure ACTUAL allocated memory for a list of N

In [ ]:
def measure_at_scale(counts: tuple[int, ...] = (1_000, 10_000, 100_000, 1_000_000)) -> None:
    """Use tracemalloc to measure ACTUAL allocated memory for a list of N
    instances of each class. Print a table with the saving in MB and as a
    percentage.

    Use tracemalloc, not sum(getsizeof(...)): getsizeof is per-object and
    ignores allocator overhead, shared references, and the list holding them.
    tracemalloc measures what the process really allocated, which is the number
    you actually care about.
    """
    raise NotImplementedError

---

## `measure_access_speed`

Time attribute reads, writes, and instance creation for both.

In [ ]:
def measure_access_speed() -> None:
    """Time attribute reads, writes, and instance creation for both.

    Predict the direction of each result BEFORE running. Then answer:
      - is the speed difference large enough to be a reason on its own?
      - which of the three operations differs most, and why?
    """
    raise NotImplementedError

---

## `what_breaks`

Demonstrate, with a try/except around each, the five things __slots__

In [ ]:
def what_breaks() -> None:
    """Demonstrate, with a try/except around each, the five things __slots__
    takes away:

      1. assigning a new attribute
      2. having a __dict__ at all
      3. weakref.ref(instance)          -- unless '__weakref__' is in __slots__
      4. functools.cached_property      -- it needs somewhere to store the value
      5. multiple inheritance from two classes that both define non-empty
         __slots__

    For each, print what happened, and write down whether it would matter for:
      (a) a Point in a physics engine
      (b) a User in a web application
      (c) a Node in a parser
    """
    raise NotImplementedError

---

## `with_dataclass`

Compare a hand-written __slots__ class with @dataclass(slots=True).

In [ ]:
def with_dataclass() -> None:
    """Compare a hand-written __slots__ class with @dataclass(slots=True).

    Show they use the same memory, and count the lines of code each took.
    Then say which you would write, and why the answer changed in 3.10.
    """
    raise NotImplementedError

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    measure_single_instance()
    measure_at_scale()
    measure_access_speed()
    what_breaks()
    with_dataclass()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.